# Assignment 2 LSTM

**Course:** GenAI for Software Development (CSCI 455/555)  
**Date:** March 2026  
**Instructor:** Dr. Antonio Mastropaolo  
**TA:** Md. Zahidul Haque Alvi

## Objective

In this notebook, we create a dataset of **tokenized Java methods along with their corresponding comments**, **train a LSTM model**, **Evaluate using BLEU, METEOR, BERTScore and the SIDE Metrics**, **and generate final predictions** for an LSTM Model that can generate natural language summaries for Java methods.

### Target Dataset Size
| Split | Size |
|-------|------|
| Training | 50,000 methods |
| Validation | 1,000 methods |
| Test | 1,000 methods |


### Pipeline Overview
1. Clone the repos the same way from Assignment 1
2. Parse AST and Extract JavaDoc Summaries
3. Shuffle and save splits
4. Write the embedding/tokenize script
5. Run the embedding generation
6. Build the LSTM Model
7. Train the model
8. Evaluate the model with BLEU, METEOR, BERTScore and the SIDE Metric
9. Generate final predictions

### Output Format
One tokenized method per line:
```
public void setName ( String name ) { this . name = name ; }
public int getAge ( ) { return this . age ; }
```
One tokenized summary per line:
```
Sets the users name
Gets the users age
```

In [ ]:
import os
import glob
import subprocess
import json
import random
import shutil
from pathlib import Path
import javalang
import pandas as pdymjumj, 

import glob
import subprocess
import shutil
import javalang
import requests
import json
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel
from sentence_transformers import util
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm

import sacrebleu
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score


In [2]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /home/lilyd/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:

# Configuration

CLONE_DIR = "dataset/java_repos"
OUTPUT_DIR = "dataset/cloned_repos"
DATA_DIR = "dataset/summarization"


CLASSES_PER_REPO = 20   # Java files to sample per repo
MIN_TOKENS = 10         # Minimum tokens per method
MAX_TOKENS = 512
MIN_UNIQUE_TOKENS = 10


VAL_SIZE = 1000
TEST_SIZE = 1000

# Create directories
os.makedirs(CLONE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Setup complete!")
print(f"Clone directory: {CLONE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Dataset directory: {DATA_DIR}")

Setup complete!
Clone directory: dataset/java_repos
Output directory: dataset/cloned_repos
Dataset directory: dataset/summarization


In [8]:
NUM_PAIRS_REQUIRED = 52000

In [4]:
import requests

def fetch_top_java_repos(num_repos=200, per_page=100):
    """
    Fetch top-starred Java repositories from GitHub API.
    Skips forked repos to avoid duplicate code.
    """
    repos = []
    page = 1

    while len(repos) < num_repos:
        url = "https://api.github.com/search/repositories"
        params = {
            "q": "language:java stars:>100",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break

        data = response.json()
        items = data.get("items", [])

        if not items:
            break

        for item in items:
            if item.get("fork", False):
                continue

            repos.append({
                "full_name": item["full_name"],
                "clone_url": item["clone_url"],
                "stars": item["stargazers_count"],
                "description": item.get("description", "")
            })

        page += 1

        if len(repos) >= num_repos:
            break

    return repos[:num_repos]

# Fetch repositories
print("Fetching top Java repositories from GitHub...")
repo_data = fetch_top_java_repos(num_repos=15)
df_repos = pd.DataFrame(repo_data)

print(f"\nFetched {len(df_repos)} repositories")
print(f"\nTop 10 repos by stars:")
df_repos.head(10)

Fetching top Java repositories from GitHub...

Fetched 15 repositories

Top 10 repos by stars:


,full_name,clone_url,stars,description
0,Snailclimb/JavaGuide,https://github.com/Snailclimb/JavaGuide.git,154189,Java 面试 & 后端通用面试指南，覆盖计算机基础、数据库、分布式、高并发与系统设计。准备...
1,krahets/hello-algo,https://github.com/krahets/hello-algo.git,123098,《Hello 算法》：动画图解、一键运行的数据结构与算法教程。支持简中、繁中、English...
2,GrowingGit/GitHub-Chinese-Top-Charts,https://github.com/GrowingGit/GitHub-Chinese-T...,106772,:cn: GitHub中文排行榜，各语言分设「软件 | 资料」榜单，精准定位中文好项目。各取...
3,iluwatar/java-design-patterns,https://github.com/iluwatar/java-design-patter...,93801,Design patterns implemented in Java
4,macrozheng/mall,https://github.com/macrozheng/mall.git,83108,mall项目是一套电商系统，包括前台商城系统及后台管理系统，基于Spring Boot+My...
5,spring-projects/spring-boot,https://github.com/spring-projects/spring-boot...,80283,Spring Boot helps you to create Spring-powered...
6,doocs/advanced-java,https://github.com/doocs/advanced-java.git,78893,😮 Core Interview Questions & Answers For Exper...
7,MisterBooo/LeetCodeAnimation,https://github.com/MisterBooo/LeetCodeAnimatio...,76715,Demonstrate all the questions on LeetCode in t...
8,elastic/elasticsearch,https://github.com/elastic/elasticsearch.git,76373,"Free and Open Source, Distributed, RESTful Sea..."
9,NationalSecurityAgency/ghidra,https://github.com/NationalSecurityAgency/ghid...,65613,Ghidra is a software reverse engineering (SRE)...


In [5]:
def clone_repo(clone_url, dest_dir):
    """
    Shallow clone a repository.
    Returns True if successful, False otherwise.
    """
    try:
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)

        cmd = ["git", "clone", "--depth", "1", "--quiet", clone_url, dest_dir]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print(f"  Timeout cloning {clone_url}")
        return False
    except Exception as e:
        print(f"  Error: {e}")
        return False


# Clone repositories
cloned_repos = []
failed_repos = []

print(f"Cloning {len(df_repos)} repositories...\n")

for idx, row in df_repos.iterrows():
    repo_name = row["full_name"]
    clone_url = row["clone_url"]

    safe_name = repo_name.replace("/", "_")
    dest_dir = os.path.join(CLONE_DIR, safe_name)

    print(f"[{idx+1}/{len(df_repos)}] Cloning {repo_name}...", end=" ")

    success = clone_repo(clone_url, dest_dir)

    if success:
        cloned_repos.append({
            "repo_name": repo_name,
            "local_path": dest_dir,
            "stars": row["stars"]
        })
        print("done")
    else:
        failed_repos.append(repo_name)
        print("failed")

print(f"\n\nSummary:")
print(f"  Successfully cloned: {len(cloned_repos)}")
print(f"  Failed: {len(failed_repos)}")

Cloning 15 repositories...

[1/15] Cloning Snailclimb/JavaGuide... done
[2/15] Cloning krahets/hello-algo... done
[3/15] Cloning GrowingGit/GitHub-Chinese-Top-Charts... done
[4/15] Cloning iluwatar/java-design-patterns... done
[5/15] Cloning macrozheng/mall... done
[6/15] Cloning spring-projects/spring-boot... done
[7/15] Cloning doocs/advanced-java... done
[8/15] Cloning MisterBooo/LeetCodeAnimation... done
[9/15] Cloning elastic/elasticsearch... done
[10/15] Cloning NationalSecurityAgency/ghidra... done
[11/15] Cloning TheAlgorithms/Java... done
[12/15] Cloning kdn251/interviews... done
[13/15] Cloning spring-projects/spring-framework... done
[14/15] Cloning termux/termux-app... done
[15/15] Cloning google/guava... done


Summary:
  Successfully cloned: 15
  Failed: 0


In [ ]:


# Constants for filtering
MIN_SUMMARY_WORDS = 3
MAX_SUMMARY_WORDS = 20
MIN_SUMMARY_CHARS = 10
MAX_SUMMARY_CHARS = 150  # Added max length filter
MAX_CODE_CHARS = 2000    # Prevent massive methods from bloating dataset

def is_pure_ascii(text):
    """Returns True if text contains only ASCII characters."""
    return all(ord(c) < 128 for c in text)

def extract_code_summary(file_path):
    pairs = []
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            source_code = f.read()

        # Quick if the whole file is non-ASCII heavy, skip it
        if not is_pure_ascii(source_code[:500]): 
            pass

        tree = javalang.parse.parse(source_code)
        source_lines = source_code.split('\n')

        for path, node in tree.filter(javalang.tree.MethodDeclaration):
            if not node.documentation or node.position is None:
                continue

            
            lines = [l.strip().strip('*').strip() for l in node.documentation.split('\n')]
            summary_lines = [l for l in lines if l and not l.startswith(('@', '<', '{'))]
            summary = " ".join(summary_lines)

            # Take first sentence and normalize
            summary = summary.split('.')[0].strip().lower()

            word_count = len(summary.split())
            char_count = len(summary)

            # Length Filters (Words and Characters)
            if not (MIN_SUMMARY_WORDS <= word_count <= MAX_SUMMARY_WORDS):
                continue
            if not (MIN_SUMMARY_CHARS <= char_count <= MAX_SUMMARY_CHARS):
                continue

            # ASCII/Language Filter
            if not is_pure_ascii(summary):
                continue


            start_line = node.position.line - 1
            open_braces = 0
            in_method = False
            method_lines = []

            for line in source_lines[start_line:]:
                method_lines.append(line)
                open_braces += line.count('{')
                open_braces -= line.count('}')
                if '{' in line:
                    in_method = True
                if in_method and open_braces == 0:
                    break

            method_source = " ".join(" ".join(method_lines).split())

            # ensure code isn't too long and is also ASCII
            if 0 < len(method_source) <= MAX_CODE_CHARS and is_pure_ascii(method_source):
                pairs.append((method_source, summary))

    except (RecursionError, javalang.tokenizer.LexerError, 
            javalang.parser.JavaSyntaxError, Exception):
        return []

    return pairs


all_pairs = []
print("Mining repositories for code-summary pairs...")

for repo in tqdm(cloned_repos):
    repo_dir = repo["local_path"]

    if not os.path.exists(repo_dir):
        continue

    for root, dirs, files in os.walk(repo_dir):
        for file in files:
            if file.endswith(".java"):
                file_path = os.path.join(root, file)
                all_pairs.extend(extract_code_summary(file_path))

                if len(all_pairs) >= NUM_PAIRS_REQUIRED:
                    break

        if len(all_pairs) >= NUM_PAIRS_REQUIRED:
            break

    if len(all_pairs) >= NUM_PAIRS_REQUIRED:
        break

pd.DataFrame(all_pairs, columns=["code","summary"]).to_csv("dataset.csv", index=False)

Mining repositories for code-summary pairs...


  0%|          | 0/15 [00:00<?, ?it/s]

 80%|████████  | 12/15 [14:36<03:39, 73.08s/it] 


## Save Outputs

We save: 
1. `train_code.txt`
2. `train_summary.txt`
3. `test_code.txt`
4. `test_summary.txt`
5. `val_code.txt`
6. `val_summary.txt`

In [19]:
random.shuffle(all_pairs)

train_pairs = all_pairs[:50000]
val_pairs = all_pairs[50000:51000]
test_pairs = all_pairs[51000:52000]

def save_split(pairs, prefix):
    with open(f"dataset/summarization/{prefix}_code.txt", "w", encoding="utf-8") as fc, \
         open(f"dataset/summarization/{prefix}_summary.txt", "w", encoding="utf-8") as fs:
        for code, summary in pairs:
            fc.write(code + "\n")
            fs.write(summary + "\n")

save_split(train_pairs, "train")
save_split(val_pairs, "val")
save_split(test_pairs, "test")

print("Splits generated: train (50k), val (1k), test (1k) saved to /dataset/summarization/")

Splits generated: train (50k), val (1k), test (1k) saved to /dataset/summarization/


In [20]:
%%writefile get_codet5_embeddings.py

import argparse
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

def main():
    parser = argparse.ArgumentParser(description="embedding using CodeT5+ for LSTM training.")
    parser.add_argument("--input", type=str, required=True)
    parser.add_argument("--output", type=str, required=True)
    parser.add_argument("--max_length", type=int, default=512)
    args = parser.parse_args()

    checkpoint = "Salesforce/codet5p-220m"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

    embedding_matrix = model.encoder.embed_tokens.weight.detach().clone()

    with open(args.input, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    token_ids = []
    for line in tqdm(lines, desc="Tokenizing"):
        ids = tokenizer.encode(line, truncation=True, max_length=args.max_length)
        token_ids.append(ids)

    output = {
        "token_ids": token_ids,
        "embedding_matrix": embedding_matrix,
        "tokenizer_name": checkpoint,
        "vocab_size": embedding_matrix.shape[0],
        "embedding_dim": embedding_matrix.shape[1],
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "bos_token_id": tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.pad_token_id
    }
    torch.save(output, args.output)

if __name__ == "__main__":
    main()

Writing get_codet5_embeddings.py


In [ ]:
#!python --version

Python 3.9.25


## Save the embeddings

1. `test_code.pt`
2. `train_code.pt`
3. `val_code.pt`

In [22]:
!python get_codet5_embeddings.py --input dataset/summarization/train_code.txt --output dataset/summarization/train_code.pt --max_length 128
!python get_codet5_embeddings.py --input dataset/summarization/train_summary.txt --output dataset/summarization/train_summary.pt --max_length 64

!python get_codet5_embeddings.py --input dataset/summarization/val_code.txt --output dataset/summarization/val_code.pt --max_length 128
!python get_codet5_embeddings.py --input dataset/summarization/val_summary.txt --output dataset/summarization/val_summary.pt --max_length 64

tokenizer_config.json: 1.48kB [00:00, 2.83MB/s]
vocab.json: 703kB [00:00, 1.10MB/s]
merges.txt: 294kB [00:00, 1.15MB/s]
added_tokens.json: 100%|██████████████████████| 2.00/2.00 [00:00<00:00, 43.6B/s]
special_tokens_map.json: 12.5kB [00:00, 19.2MB/s]
config.json: 100%|█████████████████████████████| 768/768 [00:00<00:00, 31.5kB/s]
pytorch_model.bin: 100%|█████████████████████| 446M/446M [00:41<00:00, 10.7MB/s]
Tokenizing: 100%|█████████████████████████| 1000/1000 [00:00<00:00, 8534.08it/s]


In [6]:

class SummarizationDataset(Dataset):
    def __init__(self, code_pt, summary_pt):
        code_data = torch.load(code_pt, weights_only=False)
        summary_data = torch.load(summary_pt, weights_only=False)
        self.code_ids = code_data["token_ids"]
        self.summary_ids = summary_data["token_ids"]

    def __len__(self):
        return len(self.code_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.code_ids[idx]), torch.tensor(self.summary_ids[idx])

# We retrieve metadata from one of the generated files to construct the collate_fn
meta = torch.load("dataset/summarization/train_code.pt", weights_only=False)
PAD_ID = meta["pad_token_id"]
EOS_ID = meta["eos_token_id"]
VOCAB_SIZE = meta["vocab_size"]
EMBED_DIM = meta["embedding_dim"]
PRETRAINED_EMBEDDINGS = meta["embedding_matrix"]

'''      "token_ids": token_ids,
        "embedding_matrix": embedding_matrix,
        "tokenizer_name": checkpoint,
        "vocab_size": embedding_matrix.shape[0],
        "embedding_dim": embedding_matrix.shape[1],
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id, '''

def collate_fn(batch):
    codes, summaries = zip(*batch)
    code_padded = pad_sequence(codes, batch_first=True, padding_value=PAD_ID)
    summary_padded = pad_sequence(summaries, batch_first=True, padding_value=PAD_ID)
    return code_padded, summary_padded

BATCH_SIZE = 64
train_loader = DataLoader(SummarizationDataset("dataset/summarization/train_code.pt", "dataset/summarization/train_summary.pt"), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SummarizationDataset("dataset/summarization/val_code.pt", "dataset/summarization/val_summary.pt"), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# LSTM Model
class LSTMSummarizer(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pretrained_embeddings, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight.data.copy_(pretrained_embeddings)
        self.embedding.weight.requires_grad = False # Freeze embeddings as per instructions

        self.encoder = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.2)

    def forward(self, src, tgt):
        embedded_src = self.dropout(self.embedding(src))
        _, (hidden, cell) = self.encoder(embedded_src)

        embedded_tgt = self.dropout(self.embedding(tgt[:, :-1]))
        output, _ = self.decoder(embedded_tgt, (hidden, cell))

        return self.fc(output)

    def generate(self, src, max_len, bos_id):
        self.eval()
        with torch.no_grad():
            embedded_src = self.embedding(src)
            _, (hidden, cell) = self.encoder(embedded_src)

            batch_size = src.shape[0]
            curr_token = torch.tensor([[bos_id]] * batch_size, device=src.device)

            outputs = []
            for _ in range(max_len):
                embedded_tgt = self.embedding(curr_token)
                out, (hidden, cell) = self.decoder(embedded_tgt, (hidden, cell))
                preds = self.fc(out)
                curr_token = preds.argmax(dim=-1)
                outputs.append(curr_token)

            return torch.cat(outputs, dim=1)

HIDDEN_DIM = 256
model = LSTMSummarizer(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, PRETRAINED_EMBEDDINGS, PAD_ID).to(DEVICE)

In [25]:
# Train Loop
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

EPOCHS = 10
best_val_loss = float('inf')
patience = 0

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for codes, summaries in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        codes, summaries = codes.to(DEVICE), summaries.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(codes, summaries)

        # Targets are shifted by 1 to predict the next word
        targets = summaries[:, 1:]
        loss = criterion(outputs.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for codes, summaries in val_loader:
            codes, summaries = codes.to(DEVICE), summaries.to(DEVICE)
            outputs = model(codes, summaries)
            targets = summaries[:, 1:]
            loss = criterion(outputs.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
            val_loss += loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {avg_val:.4f}")

    # Early Stopping Logic
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_lstm_model.pt")
        patience = 0
    else:
        patience += 1
        if patience >= 3:
            print("Early stopping triggered!")
            break

Starting Training...


Epoch 1/10: 100%|██████████| 782/782 [41:47<00:00,  3.21s/it]  


Epoch 1 | Train Loss: 5.3473 | Val Loss: 4.5850


Epoch 2/10: 100%|██████████| 782/782 [38:39<00:00,  2.97s/it]


Epoch 2 | Train Loss: 4.3525 | Val Loss: 4.1219


...

Epoch 8 | Train Loss: 3.0686 | Val Loss: 3.1877


Epoch 9/10: 100%|██████████| 782/782 [40:28<00:00,  3.11s/it]


Epoch 9 | Train Loss: 2.9564 | Val Loss: 3.1246


Epoch 10/10: 100%|██████████| 782/782 [39:37<00:00,  3.04s/it]


Epoch 10 | Train Loss: 2.8539 | Val Loss: 3.0781


In [ ]:
import torch
import sacrebleu
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn
from tqdm import tqdm

SIDE_MODEL_PATH = "./side_model/"  
GENERATION_MODEL_PATH = "best_lstm_model.pt"
BATCH_SIZE = 32



def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def compute_sacrebleu(preds, refs):
    # Calculates BLEU 1 - 4 using the SacreBLEU 
    formatted_refs = [refs] # SacreBLEU expects [[ref1, ref2...]]
    results = {}
    
    for n in range(1, 5):
        # We use the BLEU class for specific ngram orders
        bleu_metric = sacrebleu.metrics.BLEU(max_ngram_order=n)
        score = bleu_metric.corpus_score(preds, formatted_refs)
        results[f"BLEU-{n}"] = score.score
    return results

# Load Models & Tokenizers

# Load the model being evaluated (LSTM)
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")

model.load_state_dict(torch.load(GENERATION_MODEL_PATH, weights_only=True, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# Load SIDE Metric Model
print(f"Loading SIDE model from {SIDE_MODEL_PATH}...")
side_tokenizer = AutoTokenizer.from_pretrained(SIDE_MODEL_PATH)
side_model = AutoModel.from_pretrained(SIDE_MODEL_PATH).to(DEVICE)
side_model.eval()

# Data Loading

with open(f"{DATA_DIR}/test_code.txt", "r") as fc, open(f"{DATA_DIR}/test_summary.txt", "r") as fs:
    test_codes = [line.strip() for line in fc]
    test_refs = [line.strip() for line in fs]

# Generate Predictions 

test_preds = []
print("Generating Predictions...")
with torch.no_grad():
    for i in tqdm(range(0, len(test_codes), BATCH_SIZE)):
        batch_codes = test_codes[i:i+BATCH_SIZE]
        inputs = tokenizer(batch_codes, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        
        generated_ids = model.generate(inputs.input_ids, max_len=20, bos_id=tokenizer.eos_token_id) 
        decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        test_preds.extend(decoded_preds)

# Run Evaluation Metrics 

print("\n" + "="*30)
print("FINAL EVALUATION RESULTS")
print("="*30)

# BLEU 1-4
bleu_results = compute_sacrebleu(test_preds, test_refs)
for k, v in bleu_results.items():
    print(f"{k}: {v:.2f}")

# METEOR
m_scores = [meteor_score([r.split()], p.split()) for r, p in zip(test_refs, test_preds)]
print(f"METEOR: {sum(m_scores)/len(m_scores):.4f}")

# BERTScore
P, R, F1 = bert_score_fn(test_preds, test_refs, lang="en", verbose=False)
print(f"BERTScore (F1): {F1.mean().item():.4f}")

# SIDE Metric Implementation
print("Calculating SIDE Score...")
side_sims = []
with torch.no_grad():
    for code, pred in zip(tqdm(test_codes, desc="SIDE"), test_preds):
        pair = [code, pred]
        inputs = side_tokenizer(pair, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
        outputs = side_model(**inputs)
        
        embeddings = mean_pooling(outputs, inputs['attention_mask'])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        # Cosine similarity between code embedding (0) and pred embedding (1)
        sim = torch.mm(embeddings[0:1], embeddings[1:2].T).item()
        side_sims.append(sim)

print(f"SIDE Score: {sum(side_sims)/len(side_sims):.4f}")

Loading SIDE model from ./side_model/...
Generating Predictions...


100%|██████████| 32/32 [00:12<00:00,  2.51it/s]



FINAL EVALUATION RESULTS
BLEU-1: 28.97
BLEU-2: 18.48
BLEU-3: 13.12
BLEU-4: 9.42
METEOR: 0.2739


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore (F1): 0.8647
Calculating SIDE Score...


SIDE: 100%|██████████| 1000/1000 [07:34<00:00,  2.20it/s]

SIDE Score: 0.6093


## Generations

Saved output
`assignment2_predictions.json`

In [ ]:
# Generate final predictions JSON
output_file = "assignment2_predictions.json"
results = []
for i in range(len(test_codes)):
    results.append({
        "id": i,
        "code": test_codes[i],
        "reference": test_refs[i],
        "prediction": test_preds[i]
    })

with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} predictions to {output_file}")

Saved 1000 predictions to assignment2_predictions.json


# Final Analysis & Conclusion

## 1. Project Summary
In this assignment, we developed a sequence-to-sequence **LSTM-based model** designed to automate the generation of natural language summaries for Java methods.

We utilized a robust dataset of **52,000 pairs** extracted from high-quality GitHub repositories. By leveraging **CodeT5+ embeddings**, the model was able to benefit from pre-trained semantic representations of code while keeping the computational overhead manageable for a recurrent architecture.

## 2. Evaluation Metric Analysis

To better understand the model's performance we evaluated it using four complementary metrics.

| Metric | Focus | Why it matters here |
|------|------|------|
| **BLEU (1-4)** | n-gram overlap | Measures literal word similarity between the prediction and the reference summary. |
| **METEOR** | Synonyms & stemming | Captures performance when the model uses different words with similar meaning (e.g., *"gets"* vs *"returns"*). |
| **BERTScore** | Semantic similarity | Uses contextual embeddings to evaluate whether the **meaning** of the summary matches the reference, even when phrasing differs. |
| **SIDE Metric** | Code-to-text alignment | Uses embeddings to measure how well the generated summary reflects the **logic of the original Java code**. |

---

## 3. Performance Results

- **Final Validation Loss:** `3.0781`
- **BLEU 1 Score:** `28.97`
- **BLEU 2 Score:** `18.48`
- **BLEU 3 Score:** `13.12`
- **BLEU 4 Score:** `9.42`
- **METEOR Score:** `0.2739`
- **BERTScore (F1):** `0.8647`
- **SIDE Score:** `0.6093`

---

## 4. Observations 

The LSTM architecture provides a strong **baseline model** for neural machine translation in the software engineering domain. While LSTMs effectively capture sequential dependencies, they can struggle with **long-range dependencies** present in complex Java methods compared to modern **Transformer-based architectures**.

### Takeaways

**Pre-trained Embeddings**

Freezing the **CodeT5+ embeddings** allows the model to start with a rich, pre-trained vocabulary of programming concepts. This improves training stability and speeds up convergence.
